Data Cleaning Pipeline for the CRMLS Sold and Listing datasets.
 
Runs the 9-step cleaning workflow in order:
  1. Standardize date columns to datetime
  2. Standardize Yes/No columns to boolean; drop low-value flag columns
  3. Remove redundant duplicate-named columns
  4. Handle missing values (drop >90%, flag 50-90% as TBD)
  5. Ensure numeric fields are properly typed
  6. Geographic data checks (lat/long/out-of-state flags)
  7. Duplicate record check (by ListingKey)
  8. Date consistency flags (no rows removed)
  9. Remove invalid numeric values
 
Not modifies the input CSVs, everything runs on df.copy(). Produces two cleaned CSVs documenting every transformation made, why, and the actual counts from this run.

# Install Package

In [1]:
import os
import re
import numpy as np
import pandas as pd

In [2]:
# Config
DATA_DIR = r"D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv"
START_MONTH = 202401
END_MONTH = 202606

# Reads the mortgage-rate-enriched datasets by default (the last step in the
# pipeline before cleaning). Set False to clean the Residential-filtered
# datasets directly instead.
USE_MORTGAGE_ENRICHED_INPUT = True
 
if USE_MORTGAGE_ENRICHED_INPUT:
    SOLD_INPUT_PATH = os.path.join(DATA_DIR, f"Sold_with_MortgageRate_{START_MONTH}_{END_MONTH}.csv")
    LISTING_INPUT_PATH = os.path.join(DATA_DIR, f"Listing_with_MortgageRate_{START_MONTH}_{END_MONTH}.csv")
else:
    SOLD_INPUT_PATH = os.path.join(DATA_DIR, f"Sold_Residential_{START_MONTH}_{END_MONTH}.csv")
    LISTING_INPUT_PATH = os.path.join(DATA_DIR, f"Listing_Residential_{START_MONTH}_{END_MONTH}.csv")
 
SOLD_OUTPUT_PATH = os.path.join(DATA_DIR, f"Sold_Cleaned_{START_MONTH}_{END_MONTH}.csv")
LISTING_OUTPUT_PATH = os.path.join(DATA_DIR, f"Listing_Cleaned_{START_MONTH}_{END_MONTH}.csv")
REPORT_PATH = os.path.join(DATA_DIR, "data_cleaning_report.md")

In [3]:
# The four columns that need to become real datetime objects (Task 1).
DATE_COLUMNS = ["CloseDate", "PurchaseContractDate", "ListingContractDate", "ContractStatusChangeDate"]
 
# The five Yes/No-style columns that need to become real booleans (Task 2).
BOOLEAN_YN_COLUMNS = ["AttachedGarageYN", "ViewYN", "PoolPrivateYN", "NewConstructionYN", "FireplaceYN"]

# Columns to drop outright because they carry almost no information (Task 2b).
DROP_SINGLE_VALUE_COLUMNS = ["WaterfrontYN", "BasementYN"]
 
# California's approximate lat/long bounding box, used for the out-of-state
# geographic flag (Task 6). Anything outside this box isn't in California.
CA_LAT_RANGE = (32, 42)     # degrees North
CA_LON_RANGE = (-125, -114)  # degrees East (negative = West)
 
# All the raw value spellings that should map to True / False when
# standardizing the Yes/No columns in Task 2. Covers Python bools, common
# text variants, and 1/0 numeric encodings.
TRUE_VALUES = {True, "true", "True", "TRUE", "yes", "Yes", "YES", "y", "Y", "1", 1, 1.0}
FALSE_VALUES = {False, "false", "False", "FALSE", "no", "No", "NO", "n", "N", "0", 0, 0.0}

# ID/code-like columns that must NOT be auto-converted to numeric in Task 5,
# even though their values look like numbers -- converting a ZIP code or
# listing ID to float would strip leading zeros / misrepresent it as a
# quantity rather than an identifier.
NUMERIC_CONVERSION_EXCLUDE = {
    "PostalCode", "ListingId", "ListingKey", "ListingKeyNumeric",
    "BuyerAgentMlsId", "ListAgentEmail",
}

# Load Data

In [4]:
sold_raw = pd.read_csv(SOLD_INPUT_PATH, low_memory=False) # raw Sold data
listing_raw = pd.read_csv(LISTING_INPUT_PATH, low_memory=False) # raw Listing data

# Work on copies so sold_raw / listing_raw stay exactly as read from disk
sold = sold_raw.copy()
listing = listing_raw.copy()

# Summary compare cleaned row/column counts back against these untouched originals
print(f"Sold raw shape: {sold_raw.shape}")
print(f"Listing raw shape: {listing_raw.shape}")

Sold raw shape: (448033, 86)
Listing raw shape: (615739, 86)


# Task 1: Standardize Date Format

SUMMARY: Converts CloseDate, PurchaseContractDate, ListingContractDate, and ContractStatusChangeDate from plain text to real pandas datetime64 values in both datasets. This has to happen before any date math (durations, the ordering flags in Tasks 7-8) can be trusted. Uses errors='coerce' so a malformed date string becomes NaT (pandas' "missing date" marker) instead of crashing the whole script.

In [5]:
def standardize_dates(df, label):
    for col in DATE_COLUMNS: # go through each of the 4 target date columns
        if col not in df.columns: # guard: skip gracefully if a column isn't present
            print(f"- [{label}] {col} not found -- skipped.")
            continue
        before_missing = df[col].isna().sum() # how many values were already missing, before conversion
        df[col] = pd.to_datetime(df[col], errors="coerce") # do the actual text -> datetime conversion
        after_missing = df[col].isna().sum() # how many are missing/NaT after conversion
        newly_unparseable = after_missing - before_missing # any increase = values that failed to parse as dates
        print(f"- [{label}] {col}: dtype -> {df[col].dtype}. "
              f"Missing before={before_missing}, after={after_missing} "
              f"({newly_unparseable} values failed to parse and became NaT)")
    return df

sold = standardize_dates(sold, "Sold")
listing = standardize_dates(listing, "Listing")

- [Sold] CloseDate: dtype -> datetime64[ns]. Missing before=0, after=0 (0 values failed to parse and became NaT)
- [Sold] PurchaseContractDate: dtype -> datetime64[ns]. Missing before=198, after=198 (0 values failed to parse and became NaT)
- [Sold] ListingContractDate: dtype -> datetime64[ns]. Missing before=1, after=1 (0 values failed to parse and became NaT)
- [Sold] ContractStatusChangeDate: dtype -> datetime64[ns]. Missing before=589, after=589 (0 values failed to parse and became NaT)
- [Listing] CloseDate: dtype -> datetime64[ns]. Missing before=445084, after=445084 (0 values failed to parse and became NaT)
- [Listing] PurchaseContractDate: dtype -> datetime64[ns]. Missing before=326674, after=326674 (0 values failed to parse and became NaT)
- [Listing] ListingContractDate: dtype -> datetime64[ns]. Missing before=0, after=0 (0 values failed to parse and became NaT)
- [Listing] ContractStatusChangeDate: dtype -> datetime64[ns]. Missing before=7247, after=7247 (0 values failed to 

# Task 2: Standardize Boolean Columns

SUMMARY: Converts the 5 Yes/No-style columns ("AttachedGarageYN", "ViewYN", "PoolPrivateYN", "NewConstructionYN", "FireplaceYN") into a proper boolean dtype in both datasets. Before converting each column, checks that it actually only has 2 distinct non-null values. If it has more (or values that don't map to a known True/False spelling), the column is left alone and flagged for manual review rather than guessed at. Missing values are preserved as NA (pandas' nullable boolean dtype), not silently treated as False.

In [6]:
def standardize_boolean_columns(df, label, columns):
    for col in columns:  # go through each of the 5 target Yes/No columns
        if col not in df.columns: # guard: skip gracefully if a column isn't present
            print(f"- [{label}] {col} not found -- skipped.")
            continue
 
        uniques = list(df[col].dropna().unique()) # the distinct non-null values actually present
        if len(uniques) != 2: # required check before any conversion happens
            print(f"- [{label}] {col}: found {len(uniques)} unique non-null value(s) {uniques} "
                  f"-- expected exactly 2. Skipped conversion, needs manual review.")
            continue
 
        # Map every value to True/False using the known spellings above
        # anything unrecognized becomes pd.NA so it can be caught below
        mapped = df[col].map(lambda v: True if v in TRUE_VALUES else (False if v in FALSE_VALUES else pd.NA))
 
        # A value is "unmapped" if it was non-null in the original column but
        # came out as NA after mapping -- i.e. it didn't match a known spelling
        unmapped = df[col].notna() & mapped.isna()
        if unmapped.sum() > 0:
            print(f"- [{label}] {col}: {unmapped.sum()} non-null values didn't match known "
                  f"True/False patterns (values seen: {uniques}) -- skipped, needs manual review.")
            continue
 
        # 'boolean' (capital-B, pandas nullable dtype) instead of plain bool
        # because plain numpy bool can't represent missing values
        df[col] = mapped.astype("boolean")
        print(f"- [{label}] {col}: converted to boolean. Original values were {uniques}.")
    return df
 
 
sold = standardize_boolean_columns(sold, "Sold", BOOLEAN_YN_COLUMNS)
listing = standardize_boolean_columns(listing, "Listing", BOOLEAN_YN_COLUMNS)

- [Sold] AttachedGarageYN: converted to boolean. Original values were [False, True].
- [Sold] ViewYN: converted to boolean. Original values were [True, False].
- [Sold] PoolPrivateYN: converted to boolean. Original values were [False, True].
- [Sold] NewConstructionYN: converted to boolean. Original values were [False, True].
- [Sold] FireplaceYN: converted to boolean. Original values were [True, False].
- [Listing] AttachedGarageYN: converted to boolean. Original values were [True, False].
- [Listing] ViewYN not found -- skipped.
- [Listing] PoolPrivateYN not found -- skipped.
- [Listing] NewConstructionYN: converted to boolean. Original values were [False, True].
- [Listing] FireplaceYN: converted to boolean. Original values were [False, True].


SUMMARY (2b): Drops WaterfrontYN and BasementYN entirely. Both were found to have only one unique value and be almost entirely missing (~99.9% / ~98%), so they can't distinguish between records. Keeping them adds columns without adding information.

In [7]:
def drop_single_value_columns(df, label, columns):
    for col in columns: # go through WaterfrontYN, BasementYN
        if col not in df.columns: # guard: skip gracefully if not present
            print(f"- [{label}] {col} not found -- skipped.")
            continue
        n_unique = df[col].dropna().nunique() # confirm it really is (close to) single-valued
        missing_pct = df[col].isna().mean() * 100 # confirm it really is mostly missing
        df.drop(columns=[col], inplace=True) # remove the column entirely
        print(f"- [{label}] Dropped {col} (unique non-null values={n_unique}, missing={missing_pct:.1f}%)")
    return df
 
 
sold = drop_single_value_columns(sold, "Sold", DROP_SINGLE_VALUE_COLUMNS)
listing = drop_single_value_columns(listing, "Listing", DROP_SINGLE_VALUE_COLUMNS)

- [Sold] Dropped WaterfrontYN (unique non-null values=1, missing=99.9%)
- [Sold] Dropped BasementYN (unique non-null values=1, missing=98.0%)
- [Listing] WaterfrontYN not found -- skipped.
- [Listing] BasementYN not found -- skipped.


# Task 3: Remove Redundant Columns

SUMMARY: The original API extraction script had a few duplicate field names (e.g. 'ListPrice' requested twice), so pandas auto-suffixed the repeats with '.1' when reading the CSV. This function finds every column ending in '.N', compares it to its base column, and only drops it if every overlapping non-null value actually matches. If values genuinely differ, both columns are kept and flagged instead of silently discarding data.

In [8]:
def drop_redundant_duplicate_columns(df, label):
    # Find every column whose name ends in a dot followed by digits, e.g. 'ListPrice.1'.
    dup_suffix_cols = [c for c in df.columns if re.match(r".+\.\d+$", c)]
    dropped, flagged = [], []
 
    for dup_col in dup_suffix_cols:
        base_col = re.sub(r"\.\d+$", "", dup_col) # strip the '.N' suffix to get the base column name
        if base_col not in df.columns: # guard: base column must still exist to compare against
            continue
 
        a = df[base_col]
        b = df[dup_col]
 
        # If Task 1 already converted the base column to datetime, parse the
        # duplicate the same way before comparing; otherwise a datetime value 
        # vs. its own original text string would look like a "mismatch" even 
        # though the underlying data is identical
        if pd.api.types.is_datetime64_any_dtype(a):
            a_cmp = a
            b_cmp = pd.to_datetime(b, errors="coerce")
        else:
            a_cmp = a.astype(str) # compare as strings so dtype quirks don't cause false mismatches
            b_cmp = b.astype(str)
 
        overlap = a.notna() & b.notna() # only compare rows where both columns have a value
        mismatches = int((a_cmp[overlap] != b_cmp[overlap]).sum()) # count rows where the two columns disagree
 
        if mismatches == 0:
            df.drop(columns=[dup_col], inplace=True) # true duplicate -- safe to drop
            dropped.append(dup_col)
        else:
            flagged.append((dup_col, mismatches)) # values differ: keep both, flag for review
 
    if dropped:
        print(f"- [{label}] Dropped {len(dropped)} redundant duplicate columns: {dropped}")
    else:
        print(f"- [{label}] No duplicated columns found.")
    if flagged:
        print(f"- [{label}] Kept (values differ from base column, review manually): {flagged}")
    return df
 
 
sold = drop_redundant_duplicate_columns(sold, "Sold")
listing = drop_redundant_duplicate_columns(listing, "Listing")

- [Sold] No duplicated columns found.
- [Listing] Dropped 11 redundant duplicate columns: ['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'CloseDate.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


# Task 4: Handle Missing Values

SUMMARY: Computes missing-value percentage for every column. Columns over 90% missing are dropped outright (too sparse to be analytically useful and they just add noise). 

Columns in the 50-90% range are meaningfully incomplete but not necessarily useless, so they're only flagged here. No action is taken on them yet (TBD).

In [9]:
def handle_missing_values(df, label):
    miss_pct = (df.isna().mean() * 100).sort_values(ascending=False) # % missing per column, worst first
    high_missing = miss_pct[miss_pct > 90] # columns to drop
    mid_missing = miss_pct[(miss_pct >= 50) & (miss_pct <= 90)] # columns to flag only
 
    print(f"- [{label}] Columns with >90% missing, dropped ({len(high_missing)}):")
    for col, pct in high_missing.items(): # list every dropped column with its missing %
        print(f"    - {col}: {pct:.1f}% missing")
    df = df.drop(columns=list(high_missing.index)) # actually drop them
 
    print(f"- [{label}] Columns with 50-90% missing, kept / action TBD ({len(mid_missing)}):")
    for col, pct in mid_missing.items(): # list-only, no column is touched here
        print(f"    - {col}: {pct:.1f}% missing")
 
    return df, list(high_missing.index), list(mid_missing.index) # return the dropped/flagged names too, for the summary
 
 
sold, sold_dropped_high_missing, sold_mid_missing = handle_missing_values(sold, "Sold")
listing, listing_dropped_high_missing, listing_mid_missing = handle_missing_values(listing, "Listing")

- [Sold] Columns with >90% missing, dropped (13):
    - TaxYear: 100.0% missing
    - FireplacesTotal: 100.0% missing
    - TaxAnnualAmount: 100.0% missing
    - AboveGradeFinishedArea: 100.0% missing
    - ElementarySchoolDistrict: 100.0% missing
    - BusinessType: 100.0% missing
    - CoveredSpaces: 100.0% missing
    - MiddleOrJuniorSchoolDistrict: 100.0% missing
    - BelowGradeFinishedArea: 99.4% missing
    - LotSizeDimensions: 95.1% missing
    - BuilderName: 95.1% missing
    - BuildingAreaTotal: 93.0% missing
    - CoBuyerAgentFirstName: 90.8% missing
- [Sold] Columns with 50-90% missing, kept / action TBD (14):
    - BuyerAgencyCompensation: 89.7% missing
    - BuyerAgencyCompensationType: 89.7% missing
    - ElementarySchool: 86.7% missing
    - MiddleOrJuniorSchool: 86.6% missing
    - HighSchool: 82.6% missing
    - OriginatingSystemName: 80.0% missing
    - OriginatingSystemSubName: 80.0% missing
    - latfilled: 78.4% missing
    - lonfilled: 78.4% missing
    - CoListA

# Task 5: Ensure Numeric Fields Are Properly Typed

SUMMARY: Some genuinely numeric columns can end up stored as text (object) dtype, e.g. if one month's export had a stray formatting issue. This scans every text-typed column and converts it to float64 ONLY if every single non-null value successfully parses as a number. Genuinely categorical text columns are left completely alone. ID/code fields (ZIP codes, listing IDs, etc.) are explicitly excluded even if all-digit, since turning them into floats would lose meaning (e.g. a leading zero in a ZIP code).

In [10]:
def fix_numeric_types(df, label):
    converted, skipped_excluded, skipped_nonnumeric = [], [], []
 
    # Columns currently stored as text (covers both legacy 'object' dtype and
    # pandas' newer 'str' dtype, depending on pandas version)
    object_cols = [c for c in df.columns if df[c].dtype == object or str(df[c].dtype) == "str"]
 
    for col in object_cols:
        if col in NUMERIC_CONVERSION_EXCLUDE: # never auto-convert known ID/code columns
            skipped_excluded.append(col)
            continue
 
        coerced = pd.to_numeric(df[col], errors="coerce") # try converting every value to a number
        original_missing = df[col].isna().sum() # missing count before the attempt
        coerced_missing = coerced.isna().sum() # missing count after the attempt
 
        # If coercion didn't create any NEW missing values, every non-null value converted cleanly
        # it's genuinely numeric
        if coerced_missing == original_missing and coerced_missing < len(df):
            df[col] = coerced.astype("float64")
            converted.append(col)
        else:
            skipped_nonnumeric.append(col) # real text/categorical column, leave as-is
 
    print(f"- [{label}] Converted to float64: {converted}")
    print(f"- [{label}] Skipped (ID/code fields, kept as text): {skipped_excluded}")
    print(f"- [{label}] Skipped (contains non-numeric text, left as-is): {len(skipped_nonnumeric)} columns")
    print(f"- [{label}] Final dtype breakdown:")
    print(df.dtypes.value_counts().to_string())
    return df
 
 
sold = fix_numeric_types(sold, "Sold")
listing = fix_numeric_types(listing, "Listing")

- [Sold] Converted to float64: ['latfilled', 'lonfilled']
- [Sold] Skipped (ID/code fields, kept as text): ['ListAgentEmail', 'BuyerAgentMlsId', 'ListingId', 'PostalCode']
- [Sold] Skipped (contains non-numeric text, left as-is): 33 columns
- [Sold] Final dtype breakdown:
object            37
float64           22
boolean            5
datetime64[ns]     4
int64              3
- [Listing] Converted to float64: []
- [Listing] Skipped (ID/code fields, kept as text): ['ListAgentEmail', 'BuyerAgentMlsId', 'ListingId', 'PostalCode']
- [Listing] Skipped (contains non-numeric text, left as-is): 28 columns
- [Listing] Final dtype breakdown:
object            32
float64           20
datetime64[ns]     4
int64              3
boolean            3


# Task 6: Geographic Data Checks

SUMMARY: Adds three boolean flag columns to both datasets. 

lat_0_flag / long_0_flag catch coordinates that are exactly 0 or missing (0,0 almost always means the address failed to geocode, not a real location). 

out_of_state_flag catches coordinates outside California's bounding box and also flags missing coordinates as True, since a missing value can't be confirmed as being in-state.

In [11]:
def add_geo_flags(df, label):
    if "Latitude" not in df.columns or "Longitude" not in df.columns: # guard: both columns required
        print(f"- [{label}] Latitude/Longitude not found -- geo flags skipped.")
        return df
 
    df["lat_0_flag"] = (df["Latitude"] == 0) | (df["Latitude"].isna()) # True if lat is 0 or missing
    df["long_0_flag"] = (df["Longitude"] == 0) | (df["Longitude"].isna()) # True if long is 0 or missing
 
    in_ca_lat = df["Latitude"].between(*CA_LAT_RANGE) # True only where lat falls inside [32, 42]
    in_ca_lon = df["Longitude"].between(*CA_LON_RANGE) # True only where long falls inside [-125, -114]
    # .between() returns False for NaN, so a missing coordinate naturally
    # ends up flagged as out_of_state_flag=True via the negation below
    df["out_of_state_flag"] = ~(in_ca_lat & in_ca_lon)
 
    print(f"- [{label}] lat_0_flag True: {df['lat_0_flag'].sum()} ({df['lat_0_flag'].mean() * 100:.2f}%)")
    print(f"- [{label}] long_0_flag True: {df['long_0_flag'].sum()} ({df['long_0_flag'].mean() * 100:.2f}%)")
    print(f"- [{label}] out_of_state_flag True: {df['out_of_state_flag'].sum()} "
          f"({df['out_of_state_flag'].mean() * 100:.2f}%)")
    return df
 
 
sold = add_geo_flags(sold, "Sold")
listing = add_geo_flags(listing, "Listing")

- [Sold] lat_0_flag True: 7130 (1.59%)
- [Sold] long_0_flag True: 7130 (1.59%)
- [Sold] out_of_state_flag True: 7193 (1.61%)
- [Listing] lat_0_flag True: 81075 (13.17%)
- [Listing] long_0_flag True: 81075 (13.17%)
- [Listing] out_of_state_flag True: 81323 (13.21%)


# Task 7: Duplicate Records

SUMMARY: A listing can appear in more than one monthly export (e.g. an Active listing pulled in both March and April). This finds rows that match on every column except ListingKey/SourceMonth/year_month ("content duplicates"). If those matching rows also share the same ListingKey, they're the same MLS record and only the first copy is kept. If ListingKey differs, they're legitimately different listings that just happen to look alike, and every row is kept.

In [12]:
def dedupe_records(df, label):
    # Columns to IGNORE when checking for "identical content"
    # these are identifiers/provenance columns that are expected to differ (or be 
    # the very thing we're testing) rather than describing the property itself
    id_cols = {"ListingKey", "ListingKeyNumeric", "SourceMonth", "year_month"}
    compare_cols = [c for c in df.columns if c not in id_cols]
    rows_before = len(df)
 
    if "ListingKey" not in df.columns: # guard: can't apply the ListingKey rule without it
        print(f"- [{label}] ListingKey not found -- duplicate check skipped.")
        return df
 
    dup_mask = df.duplicated(subset=compare_cols, keep=False) # True for every row that has a content-match
    dup_rows = df[dup_mask] # just the rows involved in a duplicate group
 
    to_drop_indices = []
    n_true_dup_groups = 0
    n_distinct_key_groups = 0
 
    if len(dup_rows) > 0:
        # Group the duplicate-flagged rows by their shared content
        # each group is a set of rows that are identical except for ListingKey
        for _, group in dup_rows.groupby(compare_cols, dropna=False):
            if group["ListingKey"].nunique() == 1: # same ListingKey -> truly the same record
                to_drop_indices.extend(group.index[1:]) # keep the first row, mark the rest for removal
                n_true_dup_groups += 1
            else:  # different ListingKey -> legitimately different listings
                n_distinct_key_groups += 1 # nothing dropped, just counted for the report
 
    cleaned = df.drop(index=to_drop_indices).reset_index(drop=True) # remove the true-duplicate extras
    print(f"- [{label}] Rows involved in content-duplicate groups: {len(dup_rows)}")
    print(f"- [{label}] True duplicate groups (same ListingKey, collapsed to 1 record): {n_true_dup_groups}")
    print(f"- [{label}] Distinct-key groups (ListingKey differs, both/all kept): {n_distinct_key_groups}")
    print(f"- [{label}] Rows dropped: {len(to_drop_indices)}")
    print(f"- [{label}] Rows before: {rows_before}, after: {len(cleaned)}")
    return cleaned
 
 
sold = dedupe_records(sold, "Sold")
listing = dedupe_records(listing, "Listing")

- [Sold] Rows involved in content-duplicate groups: 100
- [Sold] True duplicate groups (same ListingKey, collapsed to 1 record): 48
- [Sold] Distinct-key groups (ListingKey differs, both/all kept): 2
- [Sold] Rows dropped: 48
- [Sold] Rows before: 448033, after: 447985
- [Listing] Rows involved in content-duplicate groups: 2
- [Listing] True duplicate groups (same ListingKey, collapsed to 1 record): 0
- [Listing] Distinct-key groups (ListingKey differs, both/all kept): 1
- [Listing] Rows dropped: 0
- [Listing] Rows before: 615739, after: 615739


# Task 8: Date Consistency Checks

SUMMARY: The expected order for any transaction is ListingContractDate -> PurchaseContractDate -> CloseDate. This adds three boolean flag columns that catch violations of that order. Nothing is removed here. These are flags for a human to review, not a filter.

In [13]:
def add_date_consistency_flags(df, label):
    required = ["CloseDate", "ListingContractDate", "PurchaseContractDate"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        # This can happen if Task 4 dropped one of these columns for being
        # >90% missing -- the flags below still get created, just default to
        # False wherever a needed date isn't available.
        print(f"- [{label}] WARNING: {missing_cols} not present (likely dropped in Task 4 for "
              f">90% missing) -- related comparisons default to False where a date is unavailable.")
 
    # Fall back to an all-missing (NaT) series if a column isn't present, so
    # the comparisons below always have something to work with instead of
    # raising a KeyError.
    close = df["CloseDate"] if "CloseDate" in df.columns else pd.Series(pd.NaT, index=df.index)
    listing_date = df["ListingContractDate"] if "ListingContractDate" in df.columns else pd.Series(pd.NaT, index=df.index)
    purchase_date = df["PurchaseContractDate"] if "PurchaseContractDate" in df.columns else pd.Series(pd.NaT, index=df.index)
 
    # NaT comparisons (e.g. NaT < NaT, or NaT < a real date) naturally
    # evaluate to False in pandas, so missing dates don't get miscounted
    # as consistency violations.
    df["listing_after_close_flag"] = close < listing_date         # True if the sale closed BEFORE it was listed
    df["purchase_after_close_flag"] = close < purchase_date       # True if the sale closed BEFORE the offer was accepted
    purchase_before_listing = purchase_date < listing_date         # True if the offer was accepted BEFORE the listing existed
 
    # Build negative_timeline_flag by checking each condition in turn and
    # flagging a row the first time it fails ANY of them. Once a row is
    # flagged True, checking a second or third condition against it can only
    # ever leave it True -- it never gets flagged "again" or counted twice.
    negative_timeline_flag = pd.Series(False, index=df.index)          # start: nothing flagged yet
 
    negative_timeline_flag = negative_timeline_flag | (close < listing_date)     # condition 1: close before listing
    negative_timeline_flag = negative_timeline_flag | (close < purchase_date)    # condition 2: close before purchase
    negative_timeline_flag = negative_timeline_flag | purchase_before_listing     # condition 3: purchase before listing
 
    df["negative_timeline_flag"] = negative_timeline_flag
 
    print(f"- [{label}] listing_after_close_flag True: {df['listing_after_close_flag'].sum()}")
    print(f"- [{label}] purchase_after_close_flag True: {df['purchase_after_close_flag'].sum()}")
    print(f"- [{label}] negative_timeline_flag True: {df['negative_timeline_flag'].sum()} "
          f"(each row counted once, even if it violates more than one condition)")
    return df
 
 
sold = add_date_consistency_flags(sold, "Sold")
listing = add_date_consistency_flags(listing, "Listing")

- [Sold] listing_after_close_flag True: 68
- [Sold] purchase_after_close_flag True: 241
- [Sold] negative_timeline_flag True: 531 (each row counted once, even if it violates more than one condition)
- [Listing] listing_after_close_flag True: 84
- [Listing] purchase_after_close_flag True: 268
- [Listing] negative_timeline_flag True: 568 (each row counted once, even if it violates more than one condition)


# Task 9: Remove Invalid Numeric Values

SUMMARY: Removes rows with impossible numeric values (e.g. a $0 close price, negative bedroom count) since these are data errors, not real observations, and would distort price/size statistics if left in.

Sold removes 5 conditions (ClosePrice, LivingArea, BedroomsTotal, BathroomsTotalInteger, DaysOnMarket) and separately measures but keeps ListPrice <= 0 for further discussion. 

Listing removes 4 conditions (ListPrice, LivingArea, BedroomsTotal, BathroomsTotalInteger) and intentionally ignores ClosePrice.

In [14]:
def clean_sold_invalid_values(df):
    label = "Sold"
    rows_before = len(df)
 
    # Build one boolean mask per invalid-value rule, only for columns that actually exist
    # a column could have been dropped already in Task 4
    criteria = {}
    if "ClosePrice" in df.columns:
        criteria["ClosePrice <= 0"] = df["ClosePrice"] <= 0
    else:
        print(f"- [{label}] WARNING: ClosePrice not present (likely dropped in Task 4 for >90% missing) "
              f"-- the ClosePrice <= 0 check could not run. A >90% missing ClosePrice in the Sold "
              f"dataset itself would be worth investigating, since Sold records should generally have it.")
    if "LivingArea" in df.columns:
        criteria["LivingArea <= 0"] = df["LivingArea"] <= 0
    if "BedroomsTotal" in df.columns:
        criteria["BedroomsTotal < 0"] = df["BedroomsTotal"] < 0
    if "BathroomsTotalInteger" in df.columns:
        criteria["BathroomsTotalInteger < 0"] = df["BathroomsTotalInteger"] < 0
    if "DaysOnMarket" in df.columns:
        criteria["DaysOnMarket < 0"] = df["DaysOnMarket"] < 0
 
    print(f"- [{label}] Invalid-value counts (checked before removal):")
    combined_mask = pd.Series(False, index=df.index) # will become True for any row that fails ANY rule
    for desc, mask in criteria.items():
        mask = mask.fillna(False) # a missing value isn't "invalid", don't flag NaN as True
        print(f"    - {desc}: {int(mask.sum())} rows")
        combined_mask = combined_mask | mask # OR each rule's mask into the running combined mask
 
    # ListPrice <= 0 is measured but NOT included in combined_mask
    # kept for further discussion rather than removed
    if "ClosePrice" in df.columns:
        listprice_leq0 = int((df["ListPrice"] <= 0).fillna(False).sum())
        print(f"- [{label}] ListPrice <= 0: {listprice_leq0} rows (NOT removed -- kept for further discussion)")
 
    cleaned = df[~combined_mask].reset_index(drop=True) # keep only rows that failed none of the rules
    print(f"- [{label}] Rows removed: {int(combined_mask.sum())}")
    print(f"- [{label}] Rows before: {rows_before}, after: {len(cleaned)}")
    return cleaned
 
 
def clean_listing_invalid_values(df):
    label = "Listing"
    rows_before = len(df)
 
    # Build one boolean mask per invalid-value rule, only for columns that actually exist
    # a column could have been dropped already in Task 4
    criteria = {}
    if "ListPrice" in df.columns:
        criteria["ListPrice <= 0"] = df["ListPrice"] <= 0
    if "LivingArea" in df.columns:
        criteria["LivingArea <= 0"] = df["LivingArea"] <= 0
    if "BedroomsTotal" in df.columns:
        criteria["BedroomsTotal < 0"] = df["BedroomsTotal"] < 0
    if "BathroomsTotalInteger" in df.columns:
        criteria["BathroomsTotalInteger < 0"] = df["BathroomsTotalInteger"] < 0
 
    print(f"- [{label}] Invalid-value counts (checked before removal):")
    combined_mask = pd.Series(False, index=df.index) # will become True for any row that fails ANY rule
    for desc, mask in criteria.items():
        mask = mask.fillna(False) # a missing value isn't "invalid", don't flag NaN as True
        print(f"    - {desc}: {int(mask.sum())} rows")
        combined_mask = combined_mask | mask # OR each rule's mask into the running combined mask
 
    cleaned = df[~combined_mask].reset_index(drop=True) # keep only rows that failed none of the rules
    print(f"- [{label}] Rows removed: {int(combined_mask.sum())}")
 
    # ClosePrice is intentionally NOT checked/filtered in the Listing dataset
    # most Listing rows are still-Active, so ClosePrice being missing/zero there 
    # doesn't mean anything is wrong
    print(f"- [{label}] ClosePrice intentionally ignored for the List dataset.")
    print(f"- [{label}] Rows before: {rows_before}, after: {len(cleaned)}")
    return cleaned
 
 
sold = clean_sold_invalid_values(sold)
listing = clean_listing_invalid_values(listing)

- [Sold] Invalid-value counts (checked before removal):
    - ClosePrice <= 0: 1 rows
    - LivingArea <= 0: 165 rows
    - BedroomsTotal < 0: 0 rows
    - BathroomsTotalInteger < 0: 0 rows
    - DaysOnMarket < 0: 50 rows
- [Sold] ListPrice <= 0: 0 rows (NOT removed -- kept for further discussion)


- [Sold] Rows removed: 216
- [Sold] Rows before: 447985, after: 447769
- [Listing] Invalid-value counts (checked before removal):
    - ListPrice <= 0: 0 rows
    - LivingArea <= 0: 393 rows
    - BedroomsTotal < 0: 0 rows
    - BathroomsTotalInteger < 0: 0 rows
- [Listing] Rows removed: 393
- [Listing] ClosePrice intentionally ignored for the List dataset.
- [Listing] Rows before: 615739, after: 615346


# Final Summary

Prints an overall before/after comparison (against the untouched sold_raw / listing_raw dataframes from the very start of the script), confirms final dtypes, and re-prints the geographic and date-consistency flag totals in one place before saving the cleaned CSVs

In [15]:
print("Row counts:")
print(f"- Sold: {sold_raw.shape[0]} raw -> {sold.shape[0]} cleaned")
print(f"- Listing: {listing_raw.shape[0]} raw -> {listing.shape[0]} cleaned")
 
print()
print("Column counts:")
print(f"- Sold: {sold_raw.shape[1]} raw -> {sold.shape[1]} cleaned")
print(f"- Listing: {listing_raw.shape[1]} raw -> {listing.shape[1]} cleaned")
 
print()
print("Sold final dtype breakdown:")
print(sold.dtypes.value_counts().to_string())
 
print()
print("Listing final dtype breakdown:")
print(listing.dtypes.value_counts().to_string())
 
print()
print("Geographic data quality summary:")
for label, df in [("Sold", sold), ("Listing", listing)]:
    if "out_of_state_flag" in df.columns:
        print(f"- [{label}] lat_0_flag: {df['lat_0_flag'].sum()}, long_0_flag: {df['long_0_flag'].sum()}, "
              f"out_of_state_flag: {df['out_of_state_flag'].sum()} (of {len(df)} total rows)")
 
print()
print("Date consistency flag counts:")
for label, df in [("Sold", sold), ("Listing", listing)]:
    if "negative_timeline_flag" in df.columns:
        print(f"- [{label}] listing_after_close_flag: {df['listing_after_close_flag'].sum()}, "
              f"purchase_after_close_flag: {df['purchase_after_close_flag'].sum()}, "
              f"negative_timeline_flag: {df['negative_timeline_flag'].sum()}")

Row counts:
- Sold: 448033 raw -> 447769 cleaned
- Listing: 615739 raw -> 615346 cleaned

Column counts:
- Sold: 86 raw -> 77 cleaned
- Listing: 86 raw -> 68 cleaned

Sold final dtype breakdown:
object            37
float64           22
bool               6
boolean            5
datetime64[ns]     4
int64              3

Listing final dtype breakdown:
object            32
float64           20
bool               6
datetime64[ns]     4
int64              3
boolean            3

Geographic data quality summary:
- [Sold] lat_0_flag: 7127, long_0_flag: 7127, out_of_state_flag: 7190 (of 447769 total rows)
- [Listing] lat_0_flag: 81054, long_0_flag: 81054, out_of_state_flag: 81301 (of 615346 total rows)

Date consistency flag counts:
- [Sold] listing_after_close_flag: 68, purchase_after_close_flag: 241, negative_timeline_flag: 530
- [Listing] listing_after_close_flag: 84, purchase_after_close_flag: 268, negative_timeline_flag: 567


In [16]:
# Write the two cleaned, analysis-ready datasets to disk.
sold.to_csv(SOLD_OUTPUT_PATH, index=False)
listing.to_csv(LISTING_OUTPUT_PATH, index=False)
print()
print(f"Saved cleaned datasets:\n- {SOLD_OUTPUT_PATH}\n- {LISTING_OUTPUT_PATH}")


Saved cleaned datasets:
- D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Sold_Cleaned_202401_202606.csv
- D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Listing_Cleaned_202401_202606.csv
